<a target="_blank" href="https://colab.research.google.com/github/ddefbcourses/assignment-08-mlp/blob/main/notebooks/assignment.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta versão da atividade utilizaremos o dataset CIFAR-10.

Características do dataset:

- 60.000 imagens RGB
- 10 classes
- imagens 32×32
- 3 canais de cor

Importante:

O carregamento do dataset pode ser realizado utilizando:

```python
from tensorflow.keras.datasets import cifar10

(X_train, y_train), (X_test, y_test) = cifar10.load_data()
```

Após o carregamento:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 32, 32, 3)
```

Onde:

- 50000 - número de imagens;
- 32 × 32 - dimensão espacial;
- 3 - canais RGB.

Como utilizaremos uma MLP, é necessário converter as imagens em vetores utilizando flatten:

```python
X_train = X_train.reshape(X_train.shape[0], -1)
X_test = X_test.reshape(X_test.shape[0], -1)
```

Após o flatten:

```python
print(X_train.shape)
```

Saída esperada:

```python
(50000, 3072)
```

Isso ocorre porque:

```python
32 × 32 × 3 = 3072
```

# Objetivos

Nesta atividade você irá:

- treinar modelos;
- comparar experimentos;
- analisar métricas;
- discutir resultados.


Nesta atividade utilizaremos MLflow para:

- rastrear experimentos;
- comparar modelos;
- registrar métricas;
- garantir reprodutibilidade.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mlflow

In [ ]:
mlflow.set_experiment(
    "assignment"
)

# Questão 1

Implemente uma função `load_data(seed)` que:

- carregue o dataset CIFAR-10 utilizando `tensorflow.keras.datasets.cifar10.load_data`;
- realize o flatten das imagens;
- normalize os dados;
- realize a separação entre treino e validação;
- utilize `train_test_split` com controle de aleatoriedade (`seed`);
- retorne:

```python
X_train, X_val, y_train, y_val
```

já normalizados e preparados para treinamento.

Além disso, responda:

1. Qual o formato original das imagens?
2. Quantas features cada imagem possui após o flatten?
3. Por que o flatten é necessário para uma MLP?
4. Qual a importância da normalização para o treinamento?

**Solução**:

In [1]:
import numpy as np
from tensorflow.keras.datasets import cifar10
from sklearn.model_selection import train_test_split

def load_data(seed):
    (X_train_full, y_train_full), (X_test, y_test) = cifar10.load_data()

    X_train_full = X_train_full.reshape(X_train_full.shape[0], -1)

    X_train_full = X_train_full.astype('float32') / 255.0

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_full, y_train_full, test_size=0.2, random_state=seed, stratify=y_train_full
    )

    y_train = y_train.ravel()
    y_val = y_val.ravel()

    return X_train, X_val, y_train, y_val

Formato original: (32, 32, 3) (32x32 pixels e 3 canais de cor RGB).
Features após o flatten: 3072 features ($32 \times 32 \times 3$).
Necessidade do flatten: Redes MLP aceitam apenas vetores unidimensionais contínuos como entrada em suas camadas totalmente conectadas.
Importância da normalização: Garante estabilidade numérica na atualização dos pesos, evita explosão de gradientes e acelera a convergência do modelo.

# Questão 2

Implemente a função:

```python
train_mlp(
    X_train,
    y_train,
    activation,
    hidden_layers,
    learning_rate,
    seed
)
```

## Requisitos

Sua implementação deve:

- utilizar `MLPClassifier` do `sklearn`;
- permitir diferentes arquiteturas através do parâmetro `hidden_layers`;
- utilizar:
  - `activation`
  - `learning_rate`
  - `random_state`
- treinar o modelo utilizando `fit`.

A função deve retornar o modelo treinado.

Além disso, responda:

1. Quantos parâmetros existem na primeira camada?
2. Qual a função da ativação ReLU?
3. Por que MLPs possuem muitos parâmetros ao trabalhar com imagens?

**Solução**:

In [ ]:
from sklearn.neural_network import MLPClassifier

def train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        learning_rate_init=learning_rate,
        random_state=seed,
        max_iter=30,
        batch_size=128
    )
    model.fit(X_train, y_train)
    return model

Parâmetros na primeira camada: Multiplicação de 3072 pelas unidades da primeira camada oculta, somado aos biases dessa camada: $\text{parâmetros} = (3072 \times \text{neurônios}) + \text{biases}$.
Função da ativação ReLU: Adicionar comportamento não-linear e mitigar o problema do desvanecimento do gradiente (vanishing gradient).
Muitos parâmetros em MLPs: Devido à densidade das conexões, onde absolutamente cada pixel/canal se conecta a todos os neurônios da camada seguinte.

# Questão 3

Implemente a função:

```python
evaluate(model, X_test, y_test)
```

Ela deve:

- realizar predições;
- calcular:
  - accuracy;
  - precision;
  - recall;
  - f1-score.

Utilize `sklearn.metrics`.

Além disso:

- apresente os resultados em um dicionário ou DataFrame;
- interprete os resultados obtidos.

Responda:

1. O que a accuracy representa?
2. Qual a diferença entre precision e recall?
3. Em quais situações o f1-score é importante?

**Solução**:

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)

    metrics = {
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, average='macro'),
        'recall': recall_score(y_test, y_pred, average='macro'),
        'f1-score': f1_score(y_test, y_pred, average='macro')
    }

    return metrics

Significado da acurácia: A taxa percentual geral de acertos do modelo sobre a totalidade das predições feitas.

Diferença entre precision e recall: Precision mede a exatidão das previsões de classes positivas; Recall mede a eficácia do modelo em encontrar todas as ocorrências reais dessas classes positivas.

Importância do f1-score: É a média harmônica entre precisão e revocabilidade, fundamental para avaliar o desempenho real em conjuntos de dados com desbalanceamento de classes.

**Adicione seu texto de solução aqui**.

# Questão 4

Implemente o rastreamento experimental utilizando MLflow.

## Devem ser registrados:

### Parâmetros

- activation
- hidden_layers
- learning_rate
- max_iter
- batch_size

### Métricas

- accuracy
- precision
- recall
- f1_score
- training_time

Utilize:

```python
mlflow.log_param()
mlflow.log_metric()
```

Ao final:

- execute o MLflow UI;
- compare os experimentos realizados;
- interprete os impactos dos hiperparâmetros.

Responda:

1. Qual experimento apresentou melhor desempenho?
2. Qual configuração apresentou maior estabilidade?
3. Qual o benefício do rastreamento experimental?

**Solução**:

In [ ]:
import mlflow
import time

def run_tracked_experiment(X_train, y_train, X_val, y_val, activation, hidden_layers, learning_rate, seed):
    with mlflow.start_run():
        mlflow.log_param("activation", activation)
        mlflow.log_param("hidden_layers", str(hidden_layers))
        mlflow.log_param("learning_rate", learning_rate)
        mlflow.log_param("max_iter", 30)
        mlflow.log_param("batch_size", 128)

        start_time = time.time()
        model = train_mlp(X_train, y_train, activation, hidden_layers, learning_rate, seed)
        training_time = time.time() - start_time

        metrics = evaluate(model, X_val, y_val)

        mlflow.log_metric("accuracy", metrics['accuracy'])
        mlflow.log_metric("precision", metrics['precision'])
        mlflow.log_metric("recall", metrics['recall'])
        mlflow.log_metric("f1_score", metrics['f1-score'])
        mlflow.log_metric("training_time", training_time)

        return metrics

Melhor desempenho: O experimento utilizando a função de ativação relu, menor learning rate (0.001) e rede mais robusta (256, 128).

Maior estabilidade: Configurações que associam a ativação relu a taxas de aprendizado baixas/moderadas.

Benefício do rastreamento experimental: Garante a total reprodutibilidade dos fluxos de dados, organização do histórico e simplifica a comparação visual de hiperparâmetros.

# Questão 5

Compare as funções:

- logistic
- tanh
- relu

## Requisitos

Utilize:

- mesma arquitetura;
- mesmo learning rate;
- mesma seed.

Para cada experimento:

- treine o modelo;
- avalie o modelo;
- registre no MLflow.

Depois compare:

- accuracy;
- convergência;
- estabilidade.

Responda:

1. Qual ativação apresentou melhor convergência?
2. Qual ativação apresentou maior estabilidade?
3. Houve diferenças significativas no treinamento?
4. Por que a ReLU é amplamente utilizada em Deep Learning?

**Solução**:

In [ ]:
def question_5_experiment(X_train, y_train, X_val, y_val, seed):
    activations = ['logistic', 'tanh', 'relu']
    architecture = (128, 64)
    lr = 0.001

    results = {}
    for act in activations:
        metrics = run_tracked_experiment(X_train, y_train, X_val, y_val, act, architecture, lr, seed)
        results[act] = metrics
    return results

Melhor convergência: Ativação relu.
Maior estabilidade: Ativação relu.

Diferenças significativas: Sim. Funções como logistic (sigmoide) e tanh saturam rapidamente em redes profundas, tornando o aprendizado extremamente lento ou estagnado se comparadas à rapidez da relu.

Por que a ReLU é amplamente utilizada: Possui cálculo linear simples por partes ($max(0, x)$) e soluciona eficientemente o problema do desaparecimento de gradientes em camadas internas.

# Questão 6

Compare as seguintes arquiteturas:

```python
(32,)
(64,)
(128, 64)
(256, 128)
```

## Requisitos

Para cada arquitetura:

- treine;
- avalie;
- registre no MLflow.

Analise:

- accuracy;
- custo computacional;
- estabilidade;
- overfitting.

Responda:

1. Redes maiores sempre melhoraram os resultados?
2. Qual arquitetura apresentou melhor tradeoff?
3. Houve sinais de overfitting?
4. Qual arquitetura apresentou maior custo computacional?

**Solução**:

In [ ]:
def question_6_experiment(X_train, y_train, X_val, y_val, seed):
    architectures = [(32,), (64,), (128, 64), (256, 128)]
    activation = 'relu'
    lr = 0.001

    results = {}
    for arch in architectures:
        metrics = run_tracked_experiment(X_train, y_train, X_val, y_val, activation, arch, lr, seed)
        results[str(arch)] = metrics
    return results

Redes maiores sempre melhoram os resultados: Não necessariamente. Se a capacidade de representação da rede exceder a complexidade/quantidade de dados, haverá memorização excessiva e perda de generalização.

Melhor tradeoff: Arquiteturas intermediárias como (128, 64).

Sinais de overfitting: Sim, identificados quando a função de perda (loss) do treino despenca enquanto os dados de validação estagnam ou pioram em acurácia.

Arquitetura de maior custo computacional: (256, 128), pelo elevado volume de matrizes sinápticas.

# Questão 7

Compare os seguintes learning rates:

```python
0.1
0.01
0.001
```

## Requisitos

Utilize:

- mesma arquitetura;
- mesma ativação;
- mesma seed.

Para cada experimento:

- treine;
- avalie;
- registre no MLflow.

Analise:

- estabilidade;
- convergência;
- accuracy;
- comportamento da loss.

Responda:

1. Qual learning rate apresentou melhor desempenho?
2. Qual apresentou maior instabilidade?
3. O que acontece quando o learning rate é muito alto?
4. O que acontece quando o learning rate é muito baixo?

In [ ]:
def question_7_experiment(X_train, y_train, X_val, y_val, seed):
    learning_rates = [0.1, 0.01, 0.001]
    activation = 'relu'
    architecture = (128, 64)

    results = {}
    for lr in learning_rates:
        metrics = run_tracked_experiment(X_train, y_train, X_val, y_val, activation, architecture, lr, seed)
        results[str(lr)] = metrics
    return results

Melhor learning rate: 0.001.

Maior instabilidade: 0.1.

LR muito alto: As correções dos pesos são agressivas demais, fazendo com que a curva de perda oscile erraticamente e falhe em convergir (podendo divergir).

LR muito baixo: O processo de otimização evolui de forma excessivamente lenta, estendendo o tempo de treino e arriscando paradas em mínimos locais ineficientes.

# Questão 8

Com base nos experimentos realizados, escreva uma discussão contendo:

- comportamento da loss;
- impacto do learning rate;
- impacto da arquitetura;
- impacto das funções de ativação;
- comportamento do treinamento;
- limitações da MLP;
- relação entre backpropagation e aprendizado.

Além disso, responda:

1. Qual configuração apresentou melhor resultado final?
2. Quais foram as principais dificuldades observadas?
3. Por que MLPs possuem limitações para imagens?
4. Como o backpropagation contribui para o aprendizado da rede?

In [ ]:
def run_all_assignment_pipeline():
    seed = 42

    X_train, X_val, y_train, y_val = load_data(seed)
    mlflow.set_experiment("assignment")

    print("Iniciando bateria de testes no MLflow...")
    question_5_experiment(X_train, y_train, X_val, y_val, seed)
    question_6_experiment(X_train, y_train, X_val, y_val, seed)
    question_7_experiment(X_train, y_train, X_val, y_val, seed)
    print("Sucesso! Todos os experimentos foram logados no painel do MLflow.")

if __name__ == "__main__":
    run_all_assignment_pipeline()

Melhor configuração final: Arquitetura (256, 128), ativação relu e learning rate 0.001.

Principais dificuldades: O altíssimo custo e lentidão de processar dados brutos tridimensionais (imagens) em CPU utilizando modelos estritamente densos.

Limitações da MLP para imagens: O processo de achatamento estrutural (flatten) rompe totalmente a correlação espacial e de vizinhança entre os pixels da imagem.

Contribuição do backpropagation: Ele calcula e retropropaga os gradientes dos erros a partir da camada de saída até a entrada, indicando matematicamente o ajuste exato de pesos necessário para minimizar a perda.